In [1]:
from google.colab import files
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!ls

drive  sample_data


In [3]:
%cd "/content/drive/MyDrive/Cram School Intermediate 1"
!ls

/content/drive/MyDrive/Cram School Intermediate 1
results  submission.csv  test_data.csv	train_data.csv


In [ ]:
!ls

In [4]:
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
test_df = pd.read_csv('test_data.csv')
train_df = pd.read_csv('train_data.csv')

In [7]:
train_df.head()

,id,text_content,sentiment_score
0,1102,Instead wish question treat break billion half...,-0.6707
1,1303,Yes unit whatever bring. Skill something knowl...,0.2156
2,1217,Expect improve push. Voice special billion res...,0.8601
3,1224,Respond role charge body summer audience. Wife...,0.6818
4,895,Player system quality crime treatment. Over ar...,-0.9436


In [6]:
def split_words(text):
  return len(text.split())
def split_sentences(text):
  return len(text.split('\n'))
def count_syl_word(word):
  word = word.lower()
  if not word:
    return 0
  vowels = 'aeiouy'
  count = 0
  if word[0] in vowels:
    count += 1
  for idx in range(1, len(word)):
    if word[idx] in vowels and word[idx-1] not in vowels:
      count += 1
  if word.endswith('e'): count -= 1 # in english, the ending 'e' isn't audible in the majority of cases, so we eliminate it as a syllable
  return max(1, count) # a word can't have 0 syllables

def get_total_syl(text): # Function applies on a text-based pandas series to get the total syllables
  words = [w for w in text.split() if w.strip()]
  return sum(count_syl_word(w) for w in words)

In [ ]:
# @markdown print most_common from a Counter object
Counter(train_df['text_content'][0].lower()).most_common()

In [ ]:
# @markdown print sorted letters (\n-z) from a Counter object
sorted(Counter(train_df['text_content'][0].lower()).items())

In [19]:
train_df['num_words'] = train_df['text_content'].apply(split_words)
train_df['num_sentences'] = train_df['text_content'].apply(split_sentences)
train_df['num_syl'] = train_df['text_content'].apply(get_total_syl)

In [ ]:
train_df.head()

In [20]:
num_words = train_df.pop('num_words')
train_df.insert(2, 'num_words', num_words)

num_sentences = train_df.pop('num_sentences')
train_df.insert(3, 'num_sentences', num_sentences)

num_syl = train_df.pop('num_syl')
train_df.insert(3, 'num_syl', num_syl)
train_df.head()

,id,text_content,num_words,num_syl,num_sentences,sentiment_score,fleich_score
0,1102,Instead wish question treat break billion half...,204,354,12,-0.6707,12.085882
1,1303,Yes unit whatever bring. Skill something knowl...,78,141,6,0.2156,6.001538
2,1217,Expect improve push. Voice special billion res...,202,367,15,0.8601,9.307917
3,1224,Respond role charge body summer audience. Wife...,235,401,17,0.6818,19.644118
4,895,Player system quality crime treatment. Over ar...,155,273,11,-0.9436,14.054018


In [21]:
def subt_1_vectorized(df):
    return np.clip(
        206.835
        - 1.015 * (df['num_words'] / df['num_sentences'])
        - 84.6 * (df['num_syl'] / df['num_words']),
        0, 100
    )

In [22]:
# Subtask 1
train_df['fleich_score'] = subt_1_vectorized(train_df)
S1 = train_df['fleich_score']
sub1_output = pd.DataFrame({
    'subtaskID': 1,
    'datapointID': train_df['id'],
    'answer': S1,
})
sub1_output.head()

,subtaskID,datapointID,answer
0,1,1102,42.774118
1,1,1303,40.709231
2,1,1217,39.462373
3,1,1224,48.444118
4,1,895,43.527566


In [11]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from datasets import Dataset

model_name = 'microsoft/deberta-v3-small'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=1) # num_labels=1 because we are doing a simple linear regression for a single score
device = "cuda" if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"The model runs on {device}")

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  286MB            

...

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  286MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.de

The model runs on cuda


In [59]:
train_df.head()

,id,text_content,num_words,num_syl,num_sentences,sentiment_score,fleich_score
0,1102,Instead wish question treat break billion half...,204,428,12,-0.6707,12.085882
1,1303,Yes unit whatever bring. Skill something knowl...,78,173,6,0.2156,6.001538
2,1217,Expect improve push. Voice special billion res...,202,439,15,0.8601,9.307917
3,1224,Respond role charge body summer audience. Wife...,235,481,17,0.6818,19.644118
4,895,Player system quality crime treatment. Over ar...,155,327,11,-0.9436,14.054018


In [60]:
train_df['text_content'][0]

'Instead wish question treat break billion half. Mr former citizen movie sure loss. Watch day space pay part accept reach body.\nBlue form cover base. Need tough effect before rest. Toward practice when money almost only born.\nDemocratic large while toward blue pressure know time. Why rule reason material movie evening. News nothing special of forward often include. Question because discussion cover medical look.\nPrevent wonder over cut recent argue same. Father agreement technology guess safe. Traditional question easy.\nMouth build east industry fish professional form. Walk ten give. Door reflect similar read dog. Opportunity always available owner bar hear.\nSecond run likely parent animal hospital. Have election pull most as southern film. Authority age appear provide.\nVery PM dark officer record section. Project sit even him force fast space. Thank term newspaper nearly forward plan student.\nFigure science order share. See table better speech between local.\nMedical save likel

In [54]:
# Prepare dataset in Hugging Face format
train_dataset = Dataset.from_pandas(train_df[['text_content', 'sentiment_score']].rename(columns={'sentiment_score': 'labels'}))
test_dataset  = Dataset.from_pandas(test_df[['text_content']])

def tokenizer_function(examples):
  return tokenizer(examples['text_content'], padding='max_length', truncation=True, max_length=442)

In [55]:
tokenized_train = train_dataset.map(tokenizer_function, batched=True)
tokenized_test = test_dataset.map(tokenizer_function, batched=True)

Map:   0%|          | 0/1050 [00:00<?, ? examples/s]

Map:   0%|          | 0/263 [00:00<?, ? examples/s]

In [56]:
training_args = TrainingArguments(
    output_dir='./results',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=4,
    max_grad_norm=1.0,
    lr_scheduler_type='cosine',
    logging_steps=10,
    weight_decay=0.01,
    fp16=False,
    bf16=False,
)

In [57]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
)

In [58]:
# Training
trainer.train()

Step,Training Loss
10,0.845503
20,3.025344
30,1.654327
40,13.874155
50,1252.611426


KeyboardInterrupt: 

**Obs:** The instance returned by trainer.predict() is not just a simple list of numbers, but an object named `PredictionOutput` containing useful info, such as:
- `predictions`: Predicted values
- `label_ids`: Real labels (if given in the test dataset, else `None`)
- `metrics`: Performance metrics calculated on that dataset

So, calling `.predictions` is necessary in order to isolate just **the prediction list**, ignoring metrics and labels.



In [38]:
# Subtask 2 + Prediction
y_pred = trainer.predict(tokenized_test).predictions.flatten()
S2 = y_pred

In [45]:
f"Mean on train: {train_df['sentiment_score'].mean()} Std on train: {train_df['sentiment_score'].std()}"

'Mean on train: -0.015251428571428574 Std: 0.5864914373010996'

In [42]:
train_df['sentiment_score'].std()

0.5864914373010996

In [46]:
f"Mean on train: {S2.mean()} Std on train: {S2.std()}"

'Mean on train: -0.030098455026745796 Std on train: 0.0001615427026990801'

### The transformer encoder model cant learn anything, since the text is randomly generated and doesn't follow a certain logic the model has been trained and accustomed to, so we will use a TfidfVectorizer + Ridge regression approach

In [23]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge

train_df = train_df.dropna(subset=['text_content', 'sentiment_score'])
test_df['text_content'] = test_df['text_content'].fillna('')

tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 3), sublinear_tf=True)
X_train = tfidf.fit_transform(train_df['text_content'])
X_test = tfidf.transform(test_df['text_content'])

y_train = train_df['sentiment_score']

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)

S2 = model.predict(X_test)

sub2_output = pd.DataFrame({
    'subtaskID': 2,
    'datapointID': test_df['id'],
    'answer': S2
})

print("Mean on train:", np.mean(model.predict(X_train)))
print("Std on train:", np.std(model.predict(X_train)))
print("Mean on test:", np.mean(model.predict(X_test)))
print("Std on test:", np.std(model.predict(X_test)))

Mean on train: -0.015251428571428591
Std on train: 0.27035475597577785
Mean on test: -0.019734429834748622
Std on test: 0.17605575943271262


In [69]:
sub2_output.head()

,subtaskID,datapointID,answer
0,2,383,-0.157980
1,2,1356,-0.000911
2,2,945,-0.102981
3,2,387,-0.096027
4,2,15,-0.084508


In [72]:
train_df.head()

,id,text_content,num_words,num_syl,num_sentences,sentiment_score,fleich_score
0,1102,Instead wish question treat break billion half...,204,428,12,-0.6707,12.085882
1,1303,Yes unit whatever bring. Skill something knowl...,78,173,6,0.2156,6.001538
2,1217,Expect improve push. Voice special billion res...,202,439,15,0.8601,9.307917
3,1224,Respond role charge body summer audience. Wife...,235,481,17,0.6818,19.644118
4,895,Player system quality crime treatment. Over ar...,155,327,11,-0.9436,14.054018


In [24]:
# Subtask 3 - KMeans approach
from sklearn.cluster import KMeans

all_texts = pd.concat([train_df['text_content'], test_df['text_content']], ignore_index=True)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), stop_words='english')
tfidf.fit(all_texts)

X_test = tfidf.transform(test_df['text_content'])

# Unsupervised clustering
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
test_clusters = kmeans.fit_predict(X_test)

# We have to find which cluster is 0 and which is 1 (ai_built vs not ai_built)
# Rule: AI texts often use a more repetitive vocabulary - bigger tf-idf density
cluster_0_mean = np.mean(X_test[test_clusters == 0].toarray())
cluster_1_mean = np.mean(X_test[test_clusters == 1].toarray())

if cluster_0_mean > cluster_1_mean:
  # If cluster 0  has bigger density, we assume that is the ai_built cluster (1)
  S3 = np.where(test_clusters == 0, 1, 0)
else:
  S3 = test_clusters

sub3_output = pd.DataFrame({
    'subtaskID': 3,
    'datapointID': test_df['id'],
    'answer': S3.astype(int)
})

print("Predicted test class distribution:\n", sub3_output['answer'].value_counts())

Predicted test class distribution:
 answer
1    157
0    106
Name: count, dtype: int64


In [76]:
sub3_output.head()

,subtaskID,datapointID,answer
0,3,383,0
1,3,1356,1
2,3,945,1
3,3,387,1
4,3,15,0


In [25]:
submission = pd.concat([sub1_output, sub2_output, sub3_output])
submission.to_csv('submission.csv', index=False)

In [27]:
submission

,subtaskID,datapointID,answer
0,1,1102,42.774118
1,1,1303,40.709231
2,1,1217,39.462373
3,1,1224,48.444118
4,1,895,43.527566
...,...,...,...
258,3,946,0.000000
259,3,742,1.000000
260,3,689,0.000000
261,3,730,1.000000


In [26]:
files.download('submission.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>